# 🧠 Notebook 07: CNN Model Training

**Objective:** Train the 1D Convolutional Neural Network for AQI category classification.

Architecture: Conv1D(64) → BN → ReLU → Conv1D(128) → BN → ReLU → MaxPool → Dense → Softmax

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data.data_loader import load_config
from src.preprocessing.sequence_generator import load_sequences
from src.training.train_cnn import train_cnn

config = load_config('../config/config.yaml')

# Load sequences
X_train, X_test, y_train, y_test = load_sequences(
    '../' + config['data']['sequences_path']
)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

## 7.1 Build and Train CNN

In [ ]:
cnn_model, cnn_history = train_cnn(
    X_train, y_train, X_test, y_test,
    config=config['training'],
    model_config=config['models']['cnn'],
    checkpoint_dir='../models'
)

## 7.2 Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(cnn_history.history['loss'], label='Train Loss')
axes[0].plot(cnn_history.history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('CNN — Training vs Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(cnn_history.history['accuracy'], label='Train Acc')
axes[1].plot(cnn_history.history['val_accuracy'], label='Val Acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('CNN — Training vs Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('CNN Model Training History', fontsize=14)
plt.tight_layout()
plt.savefig('../results/cnn_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 7.3 Quick Evaluation

In [ ]:
from src.evaluation.metrics import print_model_metrics
from src.evaluation.confusion_matrix import plot_confusion_matrix

y_pred_cnn = np.argmax(cnn_model.predict(X_test, verbose=0), axis=1)

cnn_metrics = print_model_metrics('CNN', y_test, y_pred_cnn)

os.makedirs('../results', exist_ok=True)
plot_confusion_matrix(y_test, y_pred_cnn, 'CNN',
                      save_path='../results/cnn_confusion_matrix.png')

# Save predictions for later comparison
np.save('../artifacts/y_pred_cnn.npy', y_pred_cnn)
print('CNN predictions saved.')

## ✅ CNN Training Complete

**Next:** Notebook 08 — LSTM Model Training